In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd

def init_driver():
    options = Options()
    options.add_argument("--start-maximized")
    # options.add_argument("--headless")  # Uncomment if needed
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)


def navigate_to_main_page(driver):
    driver.get("https://eproc.rajasthan.gov.in/")
    time.sleep(2)
    driver.find_element(By.ID, "PageLink_10").click()
    time.sleep(2)


def get_organisations(driver):
    return driver.find_elements(By.XPATH, "//a[contains(@id,'DirectLink')]")


def get_tenders(driver):
    return driver.find_elements(By.XPATH, "//a[contains(@id,'DirectLink')]")


def get_field_text_from_popup(driver, field_name):
    """
    Extract field text from inside the popup modal.
    Adjust xpath if popup structure differs.
    """
    try:
        xpath = f"//td/b[normalize-space()='{field_name}']/ancestor::td/following-sibling::td[1]"
        element = driver.find_element(By.XPATH, xpath)
        return element.text.strip()
    except NoSuchElementException:
        return "Not Found"


def handle_popup_and_extract_fields(driver, fields_to_extract):
    """
    Wait for popup, extract fields, close popup.
    """

    try:
        # Wait for popup/modal container to appear
        popup = WebDriverWait(driver, 10).until(
            EC.visibility_of_element_located((By.ID, "popup_content"))  # <-- Replace with actual popup modal ID
        )
        # Extract fields inside popup
        tender_data = {}
        for field in fields_to_extract:
            tender_data[field] = get_field_text_from_popup(driver, field)

        # Close popup: Find close button inside popup and click it
        close_btn = popup.find_element(By.CSS_SELECTOR, ".popup_close")  # <-- Replace with actual close button selector
        close_btn.click()

        # Wait until popup disappears
        WebDriverWait(driver, 5).until(EC.invisibility_of_element(popup))

        return tender_data

    except TimeoutException:
        print("Popup did not appear or timed out")
        return None
    except Exception as e:
        print(f"Error handling popup: {e}")
        return None
    
def get_org_tenders(org_name, org_id):
    driver = init_driver()
    navigate_to_main_page(driver)

    organisations = get_organisations(driver)
    num_orgs = len(organisations)
    org_element = driver.find_element(By.ID, org_id)

    org_element.click()

    tenders = get_tenders(driver)

    tender_links = [t.get_attribute("href") for t in tenders]

    fields_to_extract = [
                "Title",
                "Work Description",
                "Tender Value in ₹",
                "Location",
                "Organisation Chain",
                "Bid Submission End Date"
            ]
    
    tenders = []
    for i in tender_links:
        tender_data = {}
        driver.get(i)
        tender_data['Organisation'] = org_name
        tender_data['Tender Link'] = i
        for field in fields_to_extract:
            tender_data[field] = get_field_text_from_popup(driver, field)
            time.sleep(1)
        tenders.append(tender_data)
    return tenders

In [ ]:
org_name = "JDA"
org_id = "DirectLink_32"

tenders = get_org_tenders(org_name, org_id)

In [ ]:
df = pd.DataFrame(tenders)
import datetime
date = datetime.datetime.now().strftime("%Y-%m-%d")
df.to_csv('{org_name}_tenders_{date}.csv', index=False)

In [5]:
# Export DataFrame to Excel file (xlsx)
df.to_excel('jda_tenders_{date}.csv', index=False)

'2025-05-17'

In [4]:
now

datetime.datetime(2025, 5, 17, 16, 47, 56, 200152)